# Part 2 — GLM & Blend (Emerging Market)

Fits a Tweedie GLM (statsmodels, power=1.5) using features selected from XGBoost importance.
Uses target encoding with Bayesian shrinkage for categoricals. Blends XGBoost and GLM predictions.
Applies per-line calibration to fix systematic under-prediction.

**Key adaptations vs v4:**
- k-fold CV (5 folds) for GLM validation (same as XGBoost notebook)
- Linear predictor clipping (±20) to prevent exp() overflow on test data (was producing inf for buildings/contents)
- Higher smoothing (SMOOTHING=50 vs 30) — more shrinkage toward global mean for sparse categories
- Per-line calibration to training mean × trend factor (same as v4 — critical for Tweedie under-prediction)

In [0]:
# Step 1: Load data, XGBoost results, and prepare test data
import pandas as pd
import numpy as np
import os
import pickle

%pip install statsmodels openpyxl -q

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load processed training data
train = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    print(f'{line}: {train[line].shape}, target mean={train[line]["ultimate_claims"].mean():.4f}')

# Load test files
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))

# Load enrichment files
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Load XGBoost results
with open(os.path.join(PROC_DIR, 'xgboost_v1_results.pkl'), 'rb') as f:
    xgb_results = pickle.load(f)

xgb_importance = xgb_results['feature_importances']
xgb_oof = xgb_results['oof_predictions']
xgb_cv_scores = xgb_results['cv_scores']
xgb_test_preds = xgb_results['test_predictions']
TREND_FACTORS = xgb_results['trend_factors']

print(f'\nXGBoost CV D2: {xgb_cv_scores}')
print(f'Trend factors: {TREND_FACTORS}')
print(f'\nXGBoost importance top 5:')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: {xgb_importance[line]["feature"].head(5).tolist()}')

In [0]:
# Step 2: Prepare test data (clean + join enrichment — same as XGBoost notebook)
AGE_BAND_MAP = {'17-24': 20.5, '25-34': 29.5, '35-49': 42.0, '50-64': 57.0, '65+': 70.0}

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    for col in df.columns:
        if col.endswith('_age_band') or col == 'age_band':
            new_col = col.replace('_age_band', '_age')
            df[new_col] = df[col].map(AGE_BAND_MAP)
    if 'construction_type' in df.columns:
        df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper()
        df['construction_type'] = df['construction_type'].replace({'W': 'TIMBER', 'T': 'TIMBER'})
    if 'compulsory_excess' in df.columns and df['compulsory_excess'].nunique() <= 1:
        df.drop('compulsory_excess', axis=1, inplace=True)

# Zero-pad customer_id if needed
cust_lens = customers['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    max_len = cust_lens.index.max()
    customers['customer_id'] = customers['customer_id'].str.zfill(max_len)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    if 'customer_id' in df.columns:
        lens = df['customer_id'].str.len().value_counts()
        if lens.nunique() > 1:
            max_len = lens.index.max()
            df['customer_id'] = df['customer_id'].str.zfill(max_len)

# Join enrichment
customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df
    print(f'test_{line}: {df.shape}')

In [0]:
# Step 3: Fit Tweedie GLM using XGBoost top features (k-fold CV)
# Target encoding (Bayesian shrinkage) for ALL categoricals

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

SMOOTHING = 50  # v4 used 30 — higher for smaller data (more shrinkage)
MAX_FEATS = 20  # kept at 20 — higher values cause rank deficiency with this data size
LP_CLIP = 20  # clip linear predictor to prevent exp() overflow on test data

def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

def target_encode(train_series, train_target, apply_series, smoothing=50):
    """Bayesian target encoding: smooth category means toward prior."""
    prior = train_target.mean()
    grp = pd.DataFrame({'cat': train_series, 'target': train_target})
    stats = grp.groupby('cat')['target'].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + prior * smoothing) / (stats['count'] + smoothing)
    train_enc = train_series.map(smooth).fillna(prior)
    apply_enc = apply_series.map(smooth).fillna(prior)
    return train_enc, apply_enc

glm_models = {}
glm_cv_scores = {}
glm_oof_preds = {}
glm_test_preds = {}
glm_coefs = {}

N_FOLDS = 5

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}')
    print(f'=== {line.upper()} GLM (k-fold CV) ===')
    print(f'{"="*60}')

    train_df = train[line]
    test_df = test[line]

    # Select top MAX_FEATS features by XGBoost importance
    top_feats = xgb_importance[line]['feature'].head(MAX_FEATS).tolist()
    top_feats = [f for f in top_feats if f in train_df.columns and f not in ['ultimate_claims', 'row_id', 'policy_id', 'customer_id', 'address_id', 'period_start', 'period_end']]
    # Force Year into GLM features (captures trend, GLM can extrapolate to 2025)
    if 'Year' not in top_feats and 'Year' in train_df.columns:
        top_feats.append('Year')
        print(f'  -> Forced Year into GLM features for trend capture')
    print(f'  Selected {len(top_feats)} features from XGBoost importance')

    y_all = train_df['ultimate_claims'].values
    exp_all = train_df['exposure'].values
    exp_test = test_df['exposure'].values

    # k-fold CV
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    oof_pred = np.zeros(len(train_df))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        y_tr = y_all[tr_idx]
        y_val = y_all[val_idx]
        exp_tr = exp_all[tr_idx]
        exp_val = exp_all[val_idx]

        # Prepare features: target encode categoricals, standardize numerics
        X_tr = pd.DataFrame(index=tr_idx)
        X_val = pd.DataFrame(index=val_idx)

        for col in top_feats:
            tr_vals = train_df.iloc[tr_idx][col]
            val_vals = train_df.iloc[val_idx][col]
            is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'

            if is_cat:
                tr_enc, val_enc = target_encode(tr_vals, y_tr, val_vals, SMOOTHING)
                X_tr[col] = tr_enc.values
                X_val[col] = val_enc.values
            else:
                med = tr_vals.median()
                X_tr[col] = tr_vals.fillna(med).values
                X_val[col] = val_vals.fillna(med).values

        # Standardize
        scaler = StandardScaler()
        X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
        X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
        X_tr_s = sm.add_constant(X_tr_s, has_constant='add')
        X_val_s = sm.add_constant(X_val_s, has_constant='add')

        # Fit GLM Tweedie with log(exposure) offset
        try:
            model = sm.GLM(y_tr, X_tr_s,
                           family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
                           offset=np.log(exp_tr + 1e-10))
            result = model.fit(max_iter=100)

            eta_val = np.dot(X_val_s.values, result.params)
            eta_val = np.clip(eta_val, -LP_CLIP, LP_CLIP)
            val_pred = np.exp(eta_val + np.log(exp_val + 1e-10))
            val_pred = np.maximum(val_pred, 0)
            d2 = tweedie_d2(y_val, val_pred, p=1.5)
            fold_scores.append(d2)
            oof_pred[val_idx] = val_pred
            print(f'  Fold {fold+1}: D2 = {d2:.4f}, pred mean = {val_pred.mean():.1f} (actual {y_val.mean():.1f})')
        except Exception as e:
            print(f'  Fold {fold+1}: FAILED - {e}')
            fold_scores.append(0)
            oof_pred[val_idx] = exp_val * y_tr.mean()

    glm_cv_scores[line] = np.mean(fold_scores)
    glm_oof_preds[line] = oof_pred
    print(f'  CV mean D2: {glm_cv_scores[line]:.4f} (+/- {np.std(fold_scores):.4f})')
    print(f'  OOF pred mean: {oof_pred.mean():.4f} (actual mean: {y_all.mean():.4f})')
    print(f'  (XGBoost CV D2: {xgb_cv_scores[line]:.4f})')

    # Fit final model on all data for test predictions
    X_all_enc = pd.DataFrame(index=train_df.index)
    X_test_enc = pd.DataFrame(index=test_df.index)

    for col in top_feats:
        tr_vals = train_df[col]
        test_vals = test_df[col]
        is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'

        if is_cat:
            tr_enc, test_enc = target_encode(tr_vals, y_all, test_vals, SMOOTHING)
            X_all_enc[col] = tr_enc.values
            X_test_enc[col] = test_enc.values
        else:
            med = tr_vals.median()
            X_all_enc[col] = tr_vals.fillna(med).values
            X_test_enc[col] = test_vals.fillna(med).values

    scaler = StandardScaler()
    X_all_s = pd.DataFrame(scaler.fit_transform(X_all_enc), columns=X_all_enc.columns, index=X_all_enc.index)
    X_test_s = pd.DataFrame(scaler.transform(X_test_enc), columns=X_test_enc.columns, index=X_test_enc.index)
    X_all_s = sm.add_constant(X_all_s, has_constant='add')
    X_test_s = sm.add_constant(X_test_s, has_constant='add')

    model = sm.GLM(y_all, X_all_s,
                   family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
                   offset=np.log(exp_all + 1e-10))
    result = model.fit(max_iter=100)
    glm_models[line] = result

    # Coefficients
    coefs = pd.DataFrame({'feature': X_all_s.columns, 'coef': result.params, 'pvalue': result.pvalues})
    coefs = coefs.sort_values('coef', ascending=False)
    glm_coefs[line] = coefs
    print(f'  Top 5 coefficients:')
    print(f'  {coefs.head(5).to_string(index=False)}')

    # Test predictions (clip linear predictor to prevent exp() overflow)
    eta_test = np.dot(X_test_s.values, result.params)
    eta_test = np.clip(eta_test, -LP_CLIP, LP_CLIP)
    test_pred = np.exp(eta_test + np.log(exp_test + 1e-10))
    test_pred = np.maximum(test_pred, 0.01)
    glm_test_preds[line] = test_pred
    print(f'  Test predictions: mean={test_pred.mean():.4f}, min={test_pred.min():.4f}, max={test_pred.max():.4f}')

print(f'\n{"="*60}')
print(f'=== Summary: GLM CV D2 Scores ===')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: GLM={glm_cv_scores[line]:.4f}, XGB={xgb_cv_scores[line]:.4f}')

In [0]:
# Step 4: Blend XGBoost and GLM, optimize weights, calibrate, generate submissions

SUB_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/submissions_emerging_v1/'
os.makedirs(SUB_DIR, exist_ok=True)

# Blend on OOF predictions to get D2 and optimal weights
blend_d2 = {}
best_weights = {}

for line in ['motor', 'buildings', 'contents']:
    xgb_vp = xgb_oof[line]
    glm_vp = glm_oof_preds[line]
    y_all = train[line]['ultimate_claims'].values

    # Search for optimal XGBoost weight
    best_d2 = -999
    best_w = 0.5
    for w in np.arange(0.0, 0.81, 0.05):
        blend = w * xgb_vp + (1 - w) * glm_vp
        d2 = tweedie_d2(y_all, blend, p=1.5)
        if d2 > best_d2:
            best_d2 = d2
            best_w = w
    blend_d2[line] = best_d2
    best_weights[line] = best_w
    print(f'{line}: best XGB weight={best_w:.2f}, blend D2={best_d2:.4f} (XGB={xgb_cv_scores[line]:.4f}, GLM={glm_cv_scores[line]:.4f})')

print(f'\nAverage blend D2: {np.mean(list(blend_d2.values())):.4f}')

# Per-line calibration: scale predictions so mean = training mean
# This fixes the systematic under-prediction (same approach as v4)
CAL_TARGETS = {}
for line in ['motor', 'buildings', 'contents']:
    CAL_TARGETS[line] = train[line]['ultimate_claims'].mean()
print(f'\nCalibration targets (training mean): {CAL_TARGETS}')

# Generate calibrated blended submission
submission = sample_submission.copy()

for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    w = best_weights[line]

    # Get XGBoost and GLM test predictions
    xgb_pred = xgb_test_preds[line]
    glm_pred = np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01)
    glm_pred = np.clip(glm_pred, 0.01, 50000)

    # Apply trend factors
    xgb_pred = xgb_pred * TREND_FACTORS[line]
    glm_pred = glm_pred * TREND_FACTORS[line]

    # Blend
    blend_pred = w * xgb_pred + (1 - w) * glm_pred
    blend_pred = np.clip(blend_pred, 0.01, 50000)

    # Calibrate: scale so test mean = training mean * trend factor
    cal_target = CAL_TARGETS[line] * TREND_FACTORS[line]
    cal_factor = cal_target / blend_pred.mean()
    blend_pred = blend_pred * cal_factor
    blend_pred = np.clip(blend_pred, 0.01, 50000)

    print(f'{line}: blend mean={blend_pred.mean():.4f} (target={cal_target:.4f}, cal_factor={cal_factor:.4f})')

    pred_map = dict(zip(test_df['row_id'], blend_pred))
    mask = submission['row_id'].isin(pred_map)
    submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)

submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
sub_path = os.path.join(SUB_DIR, 'blend_calibrated_v1.csv')
submission.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} ({len(submission)} rows, mean={submission["prediction"].mean():.4f})')

In [0]:
# Step 5: Also generate XGBoost-only and GLM-only calibrated submissions for comparison

# XGBoost-only calibrated
sub_xgb = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    pred = xgb_test_preds[line] * TREND_FACTORS[line]
    cal_target = CAL_TARGETS[line] * TREND_FACTORS[line]
    cal_factor = cal_target / pred.mean()
    pred = np.clip(pred * cal_factor, 0.01, 50000)
    pred_map = dict(zip(test_df['row_id'], pred))
    mask = sub_xgb['row_id'].isin(pred_map)
    sub_xgb.loc[mask, 'prediction'] = sub_xgb.loc[mask, 'row_id'].map(pred_map)
sub_xgb['prediction'] = sub_xgb['prediction'].clip(lower=0.01, upper=50000).fillna(0.01)
sub_xgb.to_csv(os.path.join(SUB_DIR, 'xgboost_calibrated_v1.csv'), index=False)
print(f'XGBoost-only calibrated: mean={sub_xgb["prediction"].mean():.4f}')

# GLM-only calibrated
sub_glm = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    pred = np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01)
    pred = np.clip(pred, 0.01, 50000) * TREND_FACTORS[line]
    cal_target = CAL_TARGETS[line] * TREND_FACTORS[line]
    cal_factor = cal_target / pred.mean()
    pred = np.clip(pred * cal_factor, 0.01, 50000)
    pred_map = dict(zip(test_df['row_id'], pred))
    mask = sub_glm['row_id'].isin(pred_map)
    sub_glm.loc[mask, 'prediction'] = sub_glm.loc[mask, 'row_id'].map(pred_map)
sub_glm['prediction'] = sub_glm['prediction'].clip(lower=0.01, upper=50000).fillna(0.01)
sub_glm.to_csv(os.path.join(SUB_DIR, 'glm_calibrated_v1.csv'), index=False)
print(f'GLM-only calibrated: mean={sub_glm["prediction"].mean():.4f}')

# 50/50 blend calibrated
sub_5050 = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    xgb_p = xgb_test_preds[line] * TREND_FACTORS[line]
    glm_p = np.clip(np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01), 0.01, 50000) * TREND_FACTORS[line]
    pred = 0.5 * xgb_p + 0.5 * glm_p
    cal_target = CAL_TARGETS[line] * TREND_FACTORS[line]
    cal_factor = cal_target / pred.mean()
    pred = np.clip(pred * cal_factor, 0.01, 50000)
    pred_map = dict(zip(test_df['row_id'], pred))
    mask = sub_5050['row_id'].isin(pred_map)
    sub_5050.loc[mask, 'prediction'] = sub_5050.loc[mask, 'row_id'].map(pred_map)
sub_5050['prediction'] = sub_5050['prediction'].clip(lower=0.01, upper=50000).fillna(0.01)
sub_5050.to_csv(os.path.join(SUB_DIR, 'blend_5050_calibrated_v1.csv'), index=False)
print(f'50/50 blend calibrated: mean={sub_5050["prediction"].mean():.4f}')

In [0]:
# Step 6: Excel checkpoint — save all model comparison and results
EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_emerging_v1/'

with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'glm_blend_v1_results.xlsx'), engine='openpyxl') as writer:
    # Model comparison
    comparison = pd.DataFrame([
        {'line': line, 'xgb_d2': xgb_cv_scores[line], 'glm_d2': glm_cv_scores[line],
         'blend_d2': blend_d2[line], 'best_xgb_weight': best_weights[line]}
        for line in ['motor', 'buildings', 'contents']
    ])
    comparison.to_excel(writer, sheet_name='Model Comparison', index=False)

    # GLM coefficients
    for line in ['motor', 'buildings', 'contents']:
        glm_coefs[line].to_excel(writer, sheet_name=f'{line} coefs', index=False)

    # Assumptions
    assumptions = pd.DataFrame({
        'Assumption': [
            'GLM: statsmodels Tweedie (power=1.5), log link, log(exposure) offset',
            f'SMOOTHING={SMOOTHING} (v4 used 30 — higher for smaller data)',
            f'MAX_FEATS={MAX_FEATS} (v4 used 25 — fewer for smaller data)',
            'k-fold CV (5 folds) — same as XGBoost notebook',
            'Per-line calibration: test mean = training mean * trend factor',
            'Trend factors from Part 1: motor 1.04, buildings 1.096, contents 1.049',
            'Target encoding: Bayesian shrinkage, global mean prior',
        ],
    })
    assumptions.to_excel(writer, sheet_name='Assumptions', index=False)

print(f'Saved: {EXCEL_DIR}glm_blend_v1_results.xlsx')
print(f'\n=== Final Model Comparison ===')
print(comparison.to_string(index=False))
print(f'\nAverage D2: XGB={comparison["xgb_d2"].mean():.4f}, GLM={comparison["glm_d2"].mean():.4f}, Blend={comparison["blend_d2"].mean():.4f}')
print(f'\nSubmissions saved to: {SUB_DIR}')
for f in sorted(os.listdir(SUB_DIR)):
    print(f'  {f}')

## Step 7: AI/Human Decision Log

### What AI did
- Loaded processed emerging data + XGBoost results (importance, OOF predictions, test predictions)
- Prepared test data with same cleaning + enrichment joins as XGBoost notebook
- Fit Tweedie GLM (statsmodels, power=1.5) with k-fold CV (5 folds)
- Used top 20 XGBoost features for GLM (v4 used 25 — fewer for smaller data)
- Linear predictor clipping (LP_CLIP=20) to prevent exp() overflow on test data (buildings/contents were producing inf)
- Target encoding with Bayesian shrinkage (SMOOTHING=50 vs v4's 30 — more shrinkage for sparse categories)
- StandardScaler on numeric features, log(exposure) offset
- Blended XGBoost + GLM with optimized per-line weights (searched 0.0-0.8 XGB weight)
- Per-line calibration: scaled predictions so test mean = training mean * trend factor
- Generated 4 submission files: blend_calibrated, xgboost_calibrated, glm_calibrated, blend_5050_calibrated

### Key adaptations vs v4
1. **k-fold CV** for GLM validation (v4 used time-based split)
2. **SMOOTHING=50** (v4 used 30) — more shrinkage for sparse emerging data
3. **MAX_FEATS=20** (v4 used 25) — fewer features to reduce overfitting
4. **Per-line calibration** applied to all submissions (same approach as v4 — critical for Tweedie under-prediction)
5. **Trend factors from Part 1** — motor 1.04, buildings 1.096, contents 1.049

### What we tested and reverted
- **MAX_FEATS=35 + interaction features**: Tested increasing to 35 features plus domain interactions (flood_zone×sum_insured, occupation×marketing_segment, etc.). Result: SingularMatrixWarning (rank-deficient), GLM D2 dropped from 0.050 to -0.003, blend D2 from 0.061 to 0.019. Reverted to 20 features — the smaller emerging dataset (27K-47K rows per line) can't support more.
- **Interactions caused collinearity**: Categorical interactions (occupation_x_segment) created sparse categories; numeric interactions (flood_zone_x_sum_insured) were collinear with main effects. Both destabilized the GLM.

### Remaining issues
- Buildings test predictions still extreme (mean 877K) due to a few rows with very high linear predictors. LP_CLIP=20 makes them finite (485M) but downstream clip to 50000 still distorts the mean.
- Buildings cal_factor=1.42 (under-prediction) and contents cal_factor=0.52 (over-prediction) are driven by train-to-test distribution shift, not overflow.
- XGBoost D2 is negative for buildings (-0.204) and contents (-0.117) — worse than the mean model. Blend still uses 20% XGB weight as it improves the blend D2 slightly.

### Future improvements
- Use Part 1 developed claim rates as priors for target encoding (instead of global mean)
- Hyperparameter tuning for XGBoost (max_depth, eta, regularization)
- Frequency × severity decomposition to reduce variance on sparse claim rows
- Investigate test rows with extreme linear predictors — may need tighter LP_CLIP (e.g. 15) or robust target encoding caps
- Try different Tweedie power parameter (validate 1.5 is optimal for emerging data)